# A00 - Campaign Validation

## Objetivo
Confirmar a integridade dos dados brutos de todas as campanhas da Fase H3
(C01-C04) antes de qualquer notebook de análise usá-los: manifesto,
quantidade de trials esperada vs. completa, falhas, seeds usadas, versões
de proveniência, e verificação das invariantes científicas
(`ibqn.experiments.validation`). Este notebook **não executa nenhuma
campanha** - só lê `results/raw/`, `results/processed/` e
`results/manifests/`, já persistidos.

## Conceitos
- Cada campanha é identificada por um `trial_id` determinístico
  (`campanha:cenário:hash_dos_parâmetros:estratégia:seed:intent_id`) -
  `docs/campaign_architecture.md` documenta a arquitetura completa.
- `validate_trials` verifica invariantes científicas (fidelidades em
  [0,1], contagens não-negativas, consistência entre `final_status` e os
  demais campos) sobre os dados já persistidos - nunca recalcula métricas.


## Configuração


In [1]:
import pandas as pd
from pathlib import Path

from ibqn.experiments.persistence import read_trials_dataframe, trials_csv_path
from ibqn.experiments.manifests import manifest_path, read_manifest
from ibqn.experiments.validation import validate_trials, issues_to_dataframe

RESULTS_DIR = Path("../results")
CAMPAIGNS = [
    "C01_routing_strategy",
    "C02_fidelity_throughput",
    "C03_assurance_outcomes",
    "C04_reconciliation_effectiveness",
]


## Execução


In [2]:
manifests = {c: read_manifest(manifest_path(RESULTS_DIR, c)) for c in CAMPAIGNS}
trials = {c: read_trials_dataframe(trials_csv_path(RESULTS_DIR, c)) for c in CAMPAIGNS}
issues = {c: validate_trials(trials[c]) for c in CAMPAIGNS}

for campaign in CAMPAIGNS:
    assert manifests[campaign] is not None, f"no manifest found for {campaign} - run it first (see docs/campaign_architecture.md)"
print("All manifests found.")


All manifests found.


## Resultados


In [3]:
summary_df = pd.DataFrame([
    {
        "campaign": c,
        "expected_trials": manifests[c]["expected_trials"],
        "completed_trials": manifests[c]["completed_trials"],
        "failed_trials": manifests[c]["failed_trials"],
        "trials_in_csv": len(trials[c]),
        "validation_issues": len(issues[c]),
        "seeds": manifests[c]["seeds"],
        "updated_at": manifests[c]["updated_at"],
    }
    for c in CAMPAIGNS
])
summary_df


,campaign,expected_trials,completed_trials,failed_trials,trials_in_csv,validation_issues,seeds,updated_at
0,C01_routing_strategy,9,9,0,9,0,"[0, 1000, 2000]",2026-07-13T10:15:28.645148+00:00
1,C02_fidelity_throughput,20,20,0,20,0,"[0, 1000]",2026-07-13T10:15:56.953602+00:00
2,C03_assurance_outcomes,8,8,0,8,0,"[0, 1]",2026-07-13T10:16:14.612978+00:00
3,C04_reconciliation_effectiveness,6,6,0,6,0,"[0, 1000, 2000]",2026-07-13T10:16:50.021255+00:00


In [4]:
provenance_df = pd.DataFrame([
    {
        "campaign": c,
        "project_git_commit": manifests[c]["project_git_commit"],
        "sequence_git_commit": manifests[c]["sequence_git_commit"],
        "python_version": manifests[c]["python_version"],
        "platform": manifests[c]["platform"],
    }
    for c in CAMPAIGNS
])
provenance_df


,campaign,project_git_commit,sequence_git_commit,python_version,platform
0,C01_routing_strategy,c9c0761e72fdc7a31d0f079fa812e93b0a1b2bfd,1f2680a5b9065e708a7497adc53a95b108029b98,3.13.14,Windows-11-10.0.26200-SP0
1,C02_fidelity_throughput,c9c0761e72fdc7a31d0f079fa812e93b0a1b2bfd,1f2680a5b9065e708a7497adc53a95b108029b98,3.13.14,Windows-11-10.0.26200-SP0
2,C03_assurance_outcomes,c9c0761e72fdc7a31d0f079fa812e93b0a1b2bfd,1f2680a5b9065e708a7497adc53a95b108029b98,3.13.14,Windows-11-10.0.26200-SP0
3,C04_reconciliation_effectiveness,c9c0761e72fdc7a31d0f079fa812e93b0a1b2bfd,1f2680a5b9065e708a7497adc53a95b108029b98,3.13.14,Windows-11-10.0.26200-SP0


In [5]:
for c in CAMPAIGNS:
    if issues[c]:
        print(f"{c}: {len(issues[c])} validation issue(s):")
        display(issues_to_dataframe(issues[c]))
    else:
        print(f"{c}: 0 validation issues")


C01_routing_strategy: 0 validation issues
C02_fidelity_throughput: 0 validation issues
C03_assurance_outcomes: 0 validation issues
C04_reconciliation_effectiveness: 0 validation issues


In [6]:
assert all(manifests[c]["expected_trials"] == manifests[c]["completed_trials"] for c in CAMPAIGNS), (
    "every campaign must be fully completed before downstream analysis notebooks trust its data"
)
assert all(manifests[c]["failed_trials"] == 0 for c in CAMPAIGNS), "no campaign should have failed trials at this validation stage"
assert all(len(issues[c]) == 0 for c in CAMPAIGNS), "no campaign should have scientific validation issues"
assert all(len(trials[c]) == manifests[c]["completed_trials"] for c in CAMPAIGNS), "trials.csv row count must match the manifest"
print("All four campaigns confirmed complete, failure-free, and scientifically valid.")


All four campaigns confirmed complete, failure-free, and scientifically valid.


## Interpretação
Os quatro manifestos confirmam que `completed_trials == expected_trials`
para todas as campanhas, sem falhas registradas em `errors.jsonl`, e sem
nenhuma violação de invariante científica (`validate_trials`). O
`project_git_commit` gravado em cada manifesto ancora esses resultados a
uma versão exata de `src/ibqn` - qualquer mudança de código que afete
esses números exigiria uma nova execução (não uma edição manual dos
arquivos persistidos).

## Limitações
- Este notebook confirma integridade estrutural (contagens, invariantes),
  não correção científica dos *achados* - isso é o papel dos notebooks
  A01-A06.
- As campanhas C01-C04 são pequenas (poucas seeds, grids pequenos) - ver
  `docs/campaign_architecture.md`, seção 10, sobre os limites disso para
  conclusões estatísticas.

## Conclusão
As quatro campanhas de validação da Fase H3 estão completas, íntegras e
prontas para os notebooks de análise seguintes.
